
## ARIMA-Modellvergleich 

**Zeitreihen:**
Luftdruck (A): ARIMA(p_luft, 1, q_luft)  – aus Grid-Search auf Luftdruck
Wind (B): ARIMA(3, 1, 2)            – Grid-Search auf Windgeschwindigkeit
Temperatur (C): ARIMA(1, 1, 2)            – BIC-Kriterium auf Temperatur

**Evaluationsstrategie:**
1. Train/Test-Split 70/30 (Multi-Step-Prognose)
2. 5-Fold Time-Series-Cross-Validation (30 Tage pro Fold)
3. Metriken: RMSE, MAE, MAPE

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import itertools

# ---------------------------------------------------------------------------
# Repo-Stammordner automatisch finden
# Funktioniert unabhaengig vom Arbeitsverzeichnis, das VS Code setzt.
# ---------------------------------------------------------------------------
def _repo_root() -> Path:
    """Geht von der aktuellen Position aufwaerts, bis ein .git-Ordner gefunden wird."""
    p = Path().resolve()
    for folder in [p, *p.parents]:
        if (folder / ".git").is_dir():
            return folder
    return p  # Fallback: aktuelles Verzeichnis

def _find_file(folder: Path, keyword: str) -> Path:
    """
    Sucht in *folder* nach einer Datei, deren Name *keyword* enthaelt.
    Vergleich mit Unicode-Normalisierung (NFC), damit Dateien, die auf
    macOS gespeichert wurden (NFD-Umlaute), auf Windows korrekt gefunden werden.
    """
    for f in folder.iterdir():
        if keyword.lower() in unicodedata.normalize("NFC", f.name).lower():
            return f
    raise FileNotFoundError(
        f"Keine Datei mit '{keyword}' in {folder} gefunden."
    )

ROOT = _repo_root()
RAW  = ROOT / "data" / "raw"

PFAD_LUFTDRUCK  = _find_file(RAW, "Luftdruck")
PFAD_WIND       = RAW / "Wind_Zeitreihe_19470101_20241231_05705.txt"
PFAD_TEMPERATUR = RAW / "temperatur_raw_05705_akt" / "produkt_tu_termin_20241031_20260503_05705.txt"

# Pfade pruefen und anzeigen
for name, pfad in [("Luftdruck", PFAD_LUFTDRUCK), ("Wind", PFAD_WIND), ("Temperatur", PFAD_TEMPERATUR)]:
    status = "OK" if pfad.exists() else "FEHLT"
    print(f"  [{status}] {name}: {pfad.name}")

# DATEN LADEN
def lade_luftdruck(pfad):
    df = pd.read_csv(pfad, sep=";", skipinitialspace=True, na_values=["-999", "-999.0", ""])
    df.columns = df.columns.str.strip()
    df["Datum"] = pd.to_datetime(df["MESS_DATUM"].astype(str).str.strip().str[:8], format="%Y%m%d")
    df["PP_TER"] = pd.to_numeric(df["PP_TER"], errors="coerce")
    ts = df.dropna(subset=["PP_TER"]).groupby("Datum")["PP_TER"].mean().asfreq("D")
    return ts.interpolate(method="time")

def lade_wind(pfad):
    df = pd.read_csv(pfad, sep=";", skipinitialspace=True, na_values=["-999", "-999.0", ""])
    df.columns = df.columns.str.strip()
    df["Datum"] = pd.to_datetime(df["MESS_DATUM"].astype(str).str.strip().str[:8], format="%Y%m%d")
    df["FK_TER"] = pd.to_numeric(df["FK_TER"], errors="coerce")
    ts = df[df["FK_TER"] >= 0].groupby("Datum")["FK_TER"].mean().asfreq("D")
    return ts.interpolate(method="time")

def lade_temperatur(pfad):
    df = pd.read_csv(pfad, sep=";", skipinitialspace=True, na_values=["-999", "-999.0", ""])
    df.columns = df.columns.str.strip()
    df["Datum"] = pd.to_datetime(df["MESS_DATUM"].astype(str).str.strip().str[:8], format="%Y%m%d")
    df["TT_TER"] = pd.to_numeric(df["TT_TER"], errors="coerce")
    ts = df.dropna(subset=["TT_TER"]).groupby("Datum")["TT_TER"].mean().asfreq("D")
    return ts.interpolate(method="linear")

print("Lade Zeitreihen ...")
ts_luft = lade_luftdruck(PFAD_LUFTDRUCK)
ts_wind = lade_wind(PFAD_WIND)
ts_temp = lade_temperatur(PFAD_TEMPERATUR)

# MODELLE DEFINIEREN
print("\nGrid Search: Bestes Luftdruck-Modell ...")
ergebnisse = []
for p, q in itertools.product(range(4), range(4)):
    if p == 0 and q == 0:
        continue
    try:
        fit = ARIMA(ts_luft, order=(p, 1, q)).fit()
        ergebnisse.append({"p": p, "q": q, "AIC": fit.aic})
    except Exception:
        pass
erg_df = pd.DataFrame(ergebnisse).sort_values("AIC")
p_A, q_A = int(erg_df.iloc[0]["p"]), int(erg_df.iloc[0]["q"])
print(f"Bestes Luftdruck-Modell: ARIMA({p_A},1,{q_A})")

MODELLE = {
    f"Modell_A  ARIMA({p_A},1,{q_A})": (p_A, 1, q_A),
    "Modell_B  ARIMA(3,1,2)":           (3,   1, 2),
    "Modell_C  ARIMA(1,1,2)":           (1,   1, 2),
}

ZEITREIHEN = {
    "Luftdruck": ts_luft,
    "Wind":      ts_wind,
    "Temperatur": ts_temp,
}

# EVALUATIONSFUNKTIONEN
def metriken(ist, prognose):
    rmse = np.sqrt(mean_squared_error(ist, prognose))
    mae  = mean_absolute_error(ist, prognose)
    with np.errstate(divide="ignore", invalid="ignore"):
        mape = np.mean(np.abs((np.array(ist) - np.array(prognose)) / np.array(ist))) * 100
    return {"RMSE": rmse, "MAE": mae, "MAPE": mape}

def eval_train_test(ts, order):
    n = int(len(ts) * 0.70)
    train, test = ts.iloc[:n], ts.iloc[n:]
    try:
        fc = ARIMA(train, order=order).fit().get_forecast(steps=len(test)).predicted_mean
        fc.index = test.index
        return metriken(test.values, fc.values)
    except Exception:
        return {"RMSE": np.nan, "MAE": np.nan, "MAPE": np.nan}

def eval_cv(ts, order, n_splits=5, test_size=30, fenster=730):
    ts_cv = ts.iloc[-min(fenster, len(ts)):]
    tscv  = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    fold_metriken = []
    for _, (idx_tr, idx_te) in enumerate(tscv.split(ts_cv)):
        try:
            fc = ARIMA(ts_cv.iloc[idx_tr], order=order).fit() \
                     .get_forecast(steps=len(idx_te)).predicted_mean
            fc.index = ts_cv.iloc[idx_te].index
            fold_metriken.append(metriken(ts_cv.iloc[idx_te].values, fc.values))
        except Exception:
            pass
    if not fold_metriken:
        return {"RMSE": np.nan, "MAE": np.nan, "MAPE": np.nan}
    df = pd.DataFrame(fold_metriken)
    return {k: df[k].mean() for k in ["RMSE", "MAE", "MAPE"]}

# EVALUATION
print("\n" + "="*65)
print("EVALUATION: 3 Modelle x 3 Zeitreihen")
print("="*65)

records_tt, records_cv = [], []

for modell_name, order in MODELLE.items():
    for ts_name, ts in ZEITREIHEN.items():
        print(f"  {modell_name}  ×  {ts_name} ...")
        m_tt = eval_train_test(ts, order)
        m_cv = eval_cv(ts, order)
        records_tt.append({"Modell": modell_name, "Zeitreihe": ts_name, **m_tt})
        records_cv.append({"Modell": modell_name, "Zeitreihe": ts_name, **m_cv})

df_tt = pd.DataFrame(records_tt)
df_cv = pd.DataFrame(records_cv)

# ERGEBNISTABELLEN
print("\n" + "="*65)
print("RMSE – Train/Test (70/30)")
print("="*65)
print(df_tt.pivot(index="Modell", columns="Zeitreihe", values="RMSE").round(4).to_string())

print("\n" + "="*65)
print("RMSE – 5-Fold CV")
print("="*65)
print(df_cv.pivot(index="Modell", columns="Zeitreihe", values="RMSE").round(4).to_string())

print("\n" + "="*65)
print("MAE – 5-Fold CV")
print("="*65)
print(df_cv.pivot(index="Modell", columns="Zeitreihe", values="MAE").round(4).to_string())

# Rang
def rang(pivot):
    return pivot.rank(axis=0, ascending=True).astype(int)

pivot_cv = df_cv.pivot(index="Modell", columns="Zeitreihe", values="RMSE")
rang_cv  = rang(pivot_cv).mean(axis=1).sort_values()

print("\n" + "="*65)
print("MITTLERER RMSE-RANG (CV) – 1 = bestes Modell")
print("="*65)
for modell, r in rang_cv.items():
    marker = " ← EMPFOHLEN" if modell == rang_cv.index[0] else ""
    print(f"  {modell}: {r:.2f}{marker}")

## Ergebnis:
*kann man auch sehen wenn man auf "scrollable element" (blaue farbe) geht*

**Modelle**

| Kürzel | Modell | Ursprung |
|---|---|---|
| Modell A | ARIMA(3,1,1) | Grid-Search auf Luftdruck (AIC) |
| Modell B | ARIMA(3,1,2) | Grid-Search auf Wind (AIC) |
| Modell C | ARIMA(1,1,2) | BIC-Kriterium auf Temperatur |

**RMSE – Train/Test-Split (70/30, Multi-Step-Prognose)**

| Modell | Luftdruck (hPa) | Temperatur (°C) | Wind (Bft) |
|---|---|---|---|
| Modell A – ARIMA(3,1,1) | 8.5119 | 5.0538 | 0.8867 |
| Modell B – ARIMA(3,1,2) | 8.5158 | 5.0526 | 0.8869 |
| Modell C – ARIMA(1,1,2) | 8.5367 | 5.0514 | 0.8869 |

**RMSE – 5-Fold Time-Series-Cross-Validation**

| Modell | Luftdruck (hPa) | Temperatur (°C) | Wind (Bft) |
|---|---|---|---|
| Modell A – ARIMA(3,1,1) | 8.4462 | 4.2883 | 0.9181 |
| Modell B – ARIMA(3,1,2) | 8.4189 | 4.2954 | 0.9178 |
| Modell C – ARIMA(1,1,2) | 8.4358 | 4.2861 | 0.9178 |

**MAE – 5-Fold Time-Series-Cross-Validation**

| Modell | Luftdruck (hPa) | Temperatur (°C) | Wind (Bft) |
|---|---|---|---|
| Modell A – ARIMA(3,1,1) | 7.3177 | 3.5721 | 0.7313 |
| Modell B – ARIMA(3,1,2) | 7.2914 | 3.5804 | 0.7304 |
| Modell C – ARIMA(1,1,2) | 7.3123 | 3.5689 | 0.7312 |

**MAPE – 5-Fold Time-Series-Cross-Validation**

| Modell | Luftdruck (%) | Temperatur (%) | Wind (%) |
|---|---|---|---|
| Modell A – ARIMA(3,1,1) | 0.74 | 108.16 | 35.16 |
| Modell B – ARIMA(3,1,2) | 0.74 | 108.52 | 35.09 |
| Modell C – ARIMA(1,1,2) | 0.74 | 108.11 | 35.14 |

> **Hinweis MAPE Wind:** Im Train/Test-Split treten `inf`-Werte auf, da die Windzeitreihe Tage mit Windstille (Wert = 0) enthält. Division durch Null. RMSE und MAE sind für Wind die aussagekräftigeren Metriken.

**RMSE-Rang je Zeitreihe (CV) – 1 = bestes Modell**

| Modell | Luftdruck | Temperatur | Wind | Mittlerer Rang |
|---|---|---|---|---|
| Modell A – ARIMA(3,1,1) | 3 | 2 | 3 | 2.67 |
| Modell B – ARIMA(3,1,2) | 1 | 3 | 1 | 1.67 |
| **Modell C – ARIMA(1,1,2)** | **2** | **1** | **1** | **1.33 ← Empfohlen** |

**FAZIT**

**ARIMA(1,1,2) ist das robusteste Modell über alle drei Zeitreihen.**  
Es hat den niedrigsten mittleren RMSE-Rang in der Cross-Validation und ist gleichzeitig das sparsamste Modell (3 Parameter statt 5 bei Modell B). Die absoluten Unterschiede zwischen den drei Modellen sind gering – alle erfassen die Zeitreihenstruktur ähnlich gut.